Script: this piece of code runs the SEOF package on single file input for SST in the Tropical Pacific regions

This is done for observations!

Output files: Standard EOFs for Tropical Pacific

TPAC

SEOF, SPCS, FVAR, TVAR

For each model the SST fields with and without the ensemble mean removed. 
_tosDJFerem
_tosDJF'

In [2]:
#load packages needed in this notebook
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
from eofs.standard import Eof
import regionmask
import cartopy.crs as ccrs
from natsort import natsorted 

ERROR 1: PROJ: proj_create_from_database: Open of /glade/u/home/nmaher/.conda/envs/clara/share/proj failed


In [3]:
#set up the data directory and load in lon + lat + time dimensions
outputdir2='/glade/work/nmaher/SEOF_output/'
dir2='/glade/work/nmaher/obs_data/'
model = 'OBS_ERSST'
#option for single files - put one file path here to get lon/lat/time
#ds_fx = xr.open_dataset(dir2+'HadISST_1950_2018.nc_g025.nc')

ds_fx=xr.open_dataset(outputdir2+'ersstv5_1950_2015_g025.nc')

lon = ds_fx.lon
lat = ds_fx.lat
time=ds_fx.time
zg_all=ds_fx.sst






In [4]:
#select season
zg_DJF_full = zg_all.where(zg_all['time.season'] == 'DJF')

In [5]:
#take seasonal mean for masked and full regions
zg_DJF_full = zg_DJF_full.rolling(min_periods=3, center=True, time=3).mean()

In [6]:
# make annual mean
zg_DJF_full = zg_DJF_full.groupby('time.year').mean('time')

In [7]:
zg_DJF_full=np.squeeze(zg_DJF_full)

In [8]:
#remove the first time step as it is only JF not DJF
zg_DJF_2_full=zg_DJF_full[1:,:,:]
zg_DJF_2_full=zg_DJF_2_full.values

#DETREND HERE!
detrended=np.zeros([65,72,144])
from statsmodels.tsa.tsatools import detrend
for i in range(72):
    for j in range(144):
        detrended[:,i,j] = detrend(zg_DJF_2_full[0:65,i,j], order=2, axis=1)


#remove the ensemble mean to get anomalies
zg_DJF_2e_full = detrended - np.ma.average(detrended,axis=0)


In [9]:
#mask the PNA region
lat_range=[-35,35]
lon_range=[110,295]

lats=lat.values
lons=lon.values

ilat=np.logical_or(lats<-35,lats>35)
ilon = np.logical_or(lons<110,lons>295)

ilat2d=np.broadcast_to(ilat[:,np.newaxis], zg_DJF_2e_full[0,...].shape)
ilon2d=np.broadcast_to(ilon[np.newaxis,:], zg_DJF_2e_full[0,...].shape)

mask2d=np.logical_or(ilat2d,ilon2d)
mask4d=np.broadcast_to(mask2d[np.newaxis,:,:],zg_DJF_2e_full.shape)

masked_zgPNA=np.ma.masked_array(zg_DJF_2e_full,mask=mask4d)

In [10]:
#do the SEOF calculation

#weight with cosine of lat
coslat = np.cos(np.deg2rad(lat))

#how many eofs are we doing?
neof=2

#set up output dimensions
pna_seof = np.ma.masked_equal(np.zeros([neof,72,144]),0)
pna_spcs = np.ma.masked_equal(np.zeros([neof]),0)

pna_seof_fracVarExp_arr = np.ma.masked_equal(np.zeros([neof]),0)
pna_seof_totalVar_arr = np.ma.masked_equal(np.zeros([1]),0)
pna_eofs_northTest = np.ma.masked_equal(np.zeros([neof]),0)


zg_DJF_3 = masked_zgPNA.reshape(-1,72,144)

       
a=np.sqrt(coslat)
a2=a.values
wgts = np.broadcast_to(a2[np.newaxis,:,np.newaxis], zg_DJF_3.shape)

solver = Eof(zg_DJF_3, weights=wgts)


eofs = solver.eofs(neofs=neof, eofscaling=2)
pcs = solver.pcs(npcs=neof,pcscaling=1)
fracvar = solver.varianceFraction(neigs=neof)
total_variance = solver.totalAnomalyVariance()
    
pna_eofs_northTest = solver.northTest(neigs=neof, vfscaled=True)
    
spcs = np.ma.masked_array(np.zeros(pcs.shape),0)
for j in range(len(pcs[0,:])):
    spcs[:,j] = (pcs[:,j] - np.mean(pcs[:,j]))/np.std(pcs[:,j])



pna_seof = eofs
pna_spcs = spcs
pna_seof_fracVarExp_arr = fracvar
pna_seof_totalVar_arr = total_variance


In [11]:
#save output
eof_T='_TPAC_SST'

np.savez_compressed(outputdir2+model+eof_T+'_SEOF.npz', data=pna_seof.data, mask=pna_seof.mask)
np.savez_compressed(outputdir2+model+eof_T+'_SPCS.npz', data=pna_spcs.data, mask=pna_spcs.mask)
np.savez_compressed(outputdir2+model+eof_T+'_FVAR.npz', data=pna_seof_fracVarExp_arr.data)
np.savez_compressed(outputdir2+model+eof_T+'_TVAR.npz', data=pna_seof_totalVar_arr.data)
